# Training Results Analysis
Load and visualize metrics from a completed training run.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Point to your training run
RUN_DIR = Path('../runs/train/surgical_yolov8')
WEIGHTS = RUN_DIR / 'weights/best.pt'
RESULTS_CSV = RUN_DIR / 'results.csv'

In [ ]:
# Plot training curves
df = pd.read_csv(RESULTS_CSV)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
metrics = [
    ('train/box_loss', 'Train Box Loss'),
    ('train/cls_loss', 'Train Class Loss'),
    ('metrics/mAP50(B)', 'mAP@50'),
    ('metrics/mAP50-95(B)', 'mAP@50-95'),
    ('metrics/precision(B)', 'Precision'),
    ('metrics/recall(B)', 'Recall'),
]
for ax, (col, title) in zip(axes.flat, metrics):
    if col in df.columns:
        ax.plot(df['epoch'], df[col])
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_DIR / 'training_curves.png', dpi=150)
plt.show()

best_epoch = df['metrics/mAP50(B)'].idxmax()
print(f"Best epoch: {best_epoch}")
print(df.loc[best_epoch, ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(B)', 'metrics/recall(B)']])

In [ ]:
# Per-class metrics from validation
model = YOLO(str(WEIGHTS))
# Update data path as needed
metrics = model.val(data='../configs/data_endovis.yaml', split='test', device='0')

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision:{metrics.box.mp:.4f}")
print(f"Recall:   {metrics.box.mr:.4f}")